In [1]:
import numpy as np
import json
import random
import copy

In [ ]:
def calculate_greedy_cost(p_prev, p_curr, p_next, c=0):
    return abs(p_prev - p_curr)
# revise based on previous level

def calculate_planning_cost(p_prev, p_curr, p_next, c=0):

    dist_step_1 = abs(p_prev - p_curr)
    dist_step_2 = abs(p_curr - p_next)
    

    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    
    switch_penalty = c if (v1 * v2 < 0) else 0
    
    return dist_step_1 + dist_step_2 + switch_penalty

In [ ]:
def generate_levels(num_levels=50, screen_width=600, 
                    greedy_func=calculate_greedy_cost, 
                    planning_func=calculate_planning_cost, 
                    c=0,
                    degree_of_conflict=1.5):
    # generate levels wo regard to screen size, scale up based 
    # set doc to 1
    experiment_configs = []
    
    while len(experiment_configs) < num_levels:
        # level 1
        entry = random.randint(100, 500)
        
        # second level
        # sample two dist randomly, set near and far wo biasing
        side = random.choice([-1, 1]) 
        dist_near = random.randint(int(screen_width/30), int(screen_width/4))
        dist_far = random.randint(int(screen_width/3.5), int(screen_width/2))
        
        cand_a_x = entry + (side * dist_near)
        cand_b_x = entry - (side * dist_far)
        
        # checking, just for fun
        if not (0 < cand_a_x < screen_width and 0 < cand_b_x < screen_width):
            continue

        # third level
        h3_goal_x = cand_b_x + random.randint(-50, 50)
        if not (0 < h3_goal_x < screen_width):
            continue

        # costs
        cost_g_a = greedy_func(entry, cand_a_x, h3_goal_x, c)
        cost_p_a = planning_func(entry, cand_a_x, h3_goal_x, c)
        
        cost_g_b = greedy_func(entry, cand_b_x, h3_goal_x, c)
        cost_p_b = planning_func(entry, cand_b_x, h3_goal_x, c)

        greedy_prefers_a = (cost_g_a * degree_of_conflict < cost_g_b)
        planner_prefers_b = (cost_p_b * degree_of_conflict < cost_p_a)
        
        if greedy_prefers_a and planner_prefers_b:
            trial = {
                "trial_id": len(experiment_configs) + 1,
                "levels": [
                    {"level": 1, "holes": [entry]},
                    {"level": 2, "holes": [cand_a_x, cand_b_x]},
                    {"level": 3, "holes": [h3_goal_x]}
                ],
                "metadata": {
                    "greedy_choice": cand_a_x,
                    "planner_choice": cand_b_x,
                    "switch_cost_param": c,
                    "cost_greedy_diff": cost_g_b - cost_g_a,
                    "cost_planner_diff": cost_p_a - cost_p_b
                }
            }
            experiment_configs.append(trial)
            
    return experiment_configs

In [ ]:
levels = generate_levels(num_levels = 10000, screen_width= 1)

with open('trials_new.json', 'w') as f:
    json.dump(levels, f, indent=4)

# Run the code from here down in order to re-generate the trials 4/29

In [30]:
trials = json.load(open('../configs/default_experiment.json'))
#trials

In [45]:
new_trials = []

# skip practice trial
if len(trials) > 0:
    new_trials.append(trials[0])

def extract_replays(chunk_levels):
    extracted_paths = []
    seen_paths = set()
    
    for i in range(0, len(chunk_levels) - 2, 3):
        l1 = chunk_levels[i]
        l2 = chunk_levels[i+1]
        l3 = chunk_levels[i+2]
        
        if len(l1) == 1 and len(l2) == 2 and len(l3) == 1:
            path1_sig = (l1[0], l2[0], l3[0])
            path2_sig = (l1[0], l2[1], l3[0])
            
            if path1_sig not in seen_paths:
                seen_paths.add(path1_sig)
                extracted_paths.extend([ [l1[0]], [l2[0]], [l3[0]] ])
                
            if path2_sig not in seen_paths:
                seen_paths.add(path2_sig)
                extracted_paths.extend([ [l1[0]], [l2[1]], [l3[0]] ])
                
    return extracted_paths

# ADDED ENUMERATE HERE to track which main block we are processing
for block_idx, block in enumerate(trials[1:]):
    levels = block.get('levels', [])
    params = block.get('params', {})
    
    # split the sets of 3 evenly
    total_sets = len(levels) // 3
    third = total_sets // 3
    
    idx1 = third * 3
    idx2 = (third * 2) * 3
    
    chunks = [
        levels[0 : idx1],
        levels[idx1 : idx2],
        levels[idx2 : ]      
    ]
    
    for i, chunk_levels in enumerate(chunks):
        if not chunk_levels:
            continue
            
        split_block = {
            'params': copy.deepcopy(params),
            'levels': chunk_levels
        }

        if block_idx == 0 and i == 0:
            split_block['params']['pre_instructions'] = [
                "Now we can begin. There are 40 mazes in total.",
                "Each maze will take around two minutes to complete.",
                "Complete each maze as quickly as possible."
            ]
        else:
            if 'pre_instructions' in split_block['params']:
                del split_block['params']['pre_instructions']
            
        new_trials.append(split_block)
        
        # replays = extract_replays(chunk_levels)
        
        # if replays:
        #     replay_block = {
        #         'params': {
        #             'instructions': ['[REPLAY] Play through the specific paths from the previous section.']
        #         },
        #         'levels': replays
        #     }
        #     new_trials.append(replay_block)

In [46]:
instruction_trials = [{
    'params': {
        'pre_instructions': ['Sometimes, going to the closest hole will make you take longer to complete the level',
    'On this level, take the most effecient possible path',
    'If you take the less effecient paths, you\'ll be asked to repeat this block'],
    "instructions": [
        "Sometimes, going to the closest hole will make you take longer to complete the level",
        "On this level, take the most effecient possible path",
        "If you take the less effecient paths, you'll be asked to repeat this block"
      ]},
    'levels': [[5], [3, 10], [9]]
    }, {
    'params': {
        'pre_instructions': ['Sometimes, going to the closest hole will make you take longer to complete the level',
    'On this level, take the most effecient possible path',
    'If you take the less effecient paths, you\'ll be asked to repeat this block'],
    "instructions": [
        "Sometimes, going to the closest hole will make you take longer to complete the level",
        "On this level, take the most effecient possible path",
        "If you take the less effecient paths, you'll be asked to repeat this block"
      ]},
    'levels': [[6], [0, 10], [2]]
    }, {
    'params': {
        'pre_instructions': ['Sometimes, going to the closest hole will make you take longer to complete the level',
    'On this level, take the most effecient possible path',
    'If you take the less effecient paths, you\'ll be asked to repeat this block'],
    "instructions": [
        "Sometimes, going to the closest hole will make you take longer to complete the level",
        "On this level, take the most effecient possible path",
        "If you take the less effecient paths, you'll be asked to repeat this block"
      ]},
    'levels': [[4], [1, 11], [10]]
    },
    ]

new_trials.insert(1, instruction_trials[0])
new_trials.insert(2, instruction_trials[1])
new_trials.insert(3, instruction_trials[2])

In [47]:
new_trials[3]

{'params': {'pre_instructions': ['Sometimes, going to the closest hole will make you take longer to complete the level',
   'On this level, take the most effecient possible path',
   "If you take the less effecient paths, you'll be asked to repeat this block"],
  'instructions': ['Sometimes, going to the closest hole will make you take longer to complete the level',
   'On this level, take the most effecient possible path',
   "If you take the less effecient paths, you'll be asked to repeat this block"]},
 'levels': [[4], [1, 11], [10]]}

In [48]:
# change this to allow the patient to do more levels if they want

#new_trials1 = new_trials[0:15]
#new_trials2 = [new_trials[0]] + new_trials[15:29]
#new_trials2[1]['params'] = new_trials1[1]['params']
#del new_trials2[2]

new_trials1 = new_trials[0:44]

In [49]:
with open('../configs/short_trials_experiment.json', 'w') as f:
     json.dump(new_trials1, f, indent=2)

# with open('../configs/YFW_experiment2.json', 'w') as f:
#      json.dump(new_trials2, f, indent=2)     

In [44]:
len(new_trials1[4:44])

40

# Adding E.params.startCameraMode

In [ ]:
trials = json.load(open('../configs/short_trials_experiment.json'))

In [16]:
for i in range(4, len(trials)):
    if i % 2 == 1:
        trials[i]['params']['startCameraMode'] = 1

In [27]:
with open('../configs/short_trials_experiment.json', 'w') as f:
     json.dump(trials, f, indent=2)